# Approximate CEEI: Pareto Efficiency and Envy Relative to Oracle


This notebook is intentionally CEEI-only. It fits continuous predicted bundle scores, runs the same approximate price-search allocation with predicted and true utility, and compares Pareto-efficiency and envy-free scores across repeated held-out markets. It does not construct or calibrate preference tiers.


## 1. Install packages


In [ ]:
!python -m pip install -q -U "transformers>=4.56,<5" "accelerate>=1.2" "bitsandbytes>=0.46.1" scikit-learn scipy sentencepiece gdown

In [ ]:
import ast
import json
import random
import re
import shutil
import time
from itertools import combinations
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

SEED = 20260909
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the synthetic course-allocation data


In [ ]:
from pathlib import Path
import json
import os
import shutil
import zipfile

import gdown
import numpy as np
import pandas as pd

# Use a local dataset ZIP, or opt in to a link-shared Drive download.
ZIP_PATH = Path("fixed_tolerance_course_data.zip")
FILE_ID = os.environ.get("COURSE_DATA_DRIVE_FILE_ID", "").strip()

if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    if not FILE_ID:
        raise FileNotFoundError(
            f"Place a valid {ZIP_PATH} in the working directory or set "
            "COURSE_DATA_DRIVE_FILE_ID to the ID of a link-shared dataset."
        )

    downloaded_path = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )
    if downloaded_path is None:
        raise RuntimeError(
            "The Google Drive download failed. Check that anyone with the link "
            "can download the configured file."
        )

assert ZIP_PATH.exists(), f"Downloaded file not found: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), (
    "The downloaded file is not a valid ZIP. Check the Google Drive "
    "sharing permission and FILE_ID."
)

print(
    f"Dataset ZIP: {ZIP_PATH} "
    f"({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)"
)

# Extract the dataset into a clean input directory.
INPUT_DIR = Path("fixed_tolerance_input")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), INPUT_DIR)

# Locate the required files even if the ZIP contains a parent folder.
found = {
    path.name: path
    for path in INPUT_DIR.rglob("*")
    if path.is_file()
}

required = {
    "course_catalog.csv",
    "student_reports.csv",
    "student_ground_truth.json",
    "bundle_queries_with_truth.csv",
    "generation_metadata.json",
}

missing = required - set(found)

assert not missing, (
    f"Missing required files: {sorted(missing)}. "
    f"Files found: {sorted(found)}"
)

# Load the dataset.
catalog = pd.read_csv(found["course_catalog.csv"])
reports_df = pd.read_csv(found["student_reports.csv"])
queries = pd.read_csv(found["bundle_queries_with_truth.csv"])

metadata = json.loads(
    found["generation_metadata.json"].read_text(encoding="utf-8")
)

ground_truth = json.loads(
    found["student_ground_truth.json"].read_text(encoding="utf-8")
)

reports = (
    reports_df
    .set_index("student_id")["language_report"]
    .to_dict()
)

COURSE_IDS = sorted(
    catalog["course_id"].astype(int).tolist()
)

MAX_BUNDLE_SIZE = int(
    metadata["maximum_bundle_size"]
)

TRUE_COMMON_TOLERANCE = float(
    metadata["common_tolerance"]
)

# Convert serialized bundle lists back into Python lists.
for column in ["bundle_a", "bundle_b"]:
    queries[column] = queries[column].apply(json.loads)

# Student splits are generated later inside the mechanism-specific experiment.
student_ids = np.array(sorted(reports_df["student_id"].astype(int).unique()))
print("Students:", len(student_ids))
print("Queries:", len(queries))
print("True tolerance (used only to recover ordinal ground-truth tiers):", TRUE_COMMON_TOLERANCE)

display(catalog)
display(reports_df.sample(3, random_state=SEED))
display(queries.groupby("true_label").size().rename("count"))


## 3. Load the open-source LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))

## 4. Structured profile prompt and robust validation

In [ ]:
PUBLIC_CATALOG = catalog[
    ["course_id", "course_name", "field", "time_slot", "workload", "description"]
].to_dict("records")

PROFILE_PROMPT = '''
You extract a structured course-preference profile from a student's report.
Use only the report and the public course catalog. Never invent preferences.

For every course with clear evidence, assign exactly one attitude from:
STRONGLY_DISLIKE, DISLIKE, NEUTRAL, LIKE, STRONGLY_LIKE.
Omit courses without clear evidence; omitted courses are treated as NEUTRAL.

Extract complements only when two courses are especially valuable together.
Extract substitutes only when the student indicates overlap, redundancy, or a
desire not to take both. Represent a pair as "smaller_id|larger_id".

Return exactly one JSON object:
{
  "course_attitudes": [{"course_id": 3, "attitude": "LIKE"}],
  "complements": ["0|6"],
  "substitutes": ["2|7"]
}
Do not return Markdown, explanations, confidence scores, tolerance estimates,
or additional keys.
'''.strip()

ATTITUDE_VALUES = {
    "STRONGLY_DISLIKE": -2.0,
    "DISLIKE": -1.0,
    "NEUTRAL": 0.0,
    "LIKE": 1.0,
    "STRONGLY_LIKE": 2.0,
}
ATTITUDE_ALIASES = {
    "STRONG_DISLIKE": "STRONGLY_DISLIKE",
    "VERY_DISLIKE": "STRONGLY_DISLIKE",
    "STRONG_LIKE": "STRONGLY_LIKE",
    "VERY_LIKE": "STRONGLY_LIKE",
}


def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end + 1])


def canonical_pair(value):
    if isinstance(value, str):
        pieces = value.split("|")
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        pieces = value
    elif isinstance(value, dict):
        pieces = [value.get("course_id_1"), value.get("course_id_2")]
    else:
        raise ValueError(f"Invalid pair: {value}")
    a, b = sorted(map(int, pieces))
    if a == b or a not in COURSE_IDS or b not in COURSE_IDS:
        raise ValueError(f"Invalid course pair: {value}")
    return (a, b)


def validate_profile(parsed):
    if not isinstance(parsed, dict):
        raise ValueError("The parsed response is not an object.")

    course_scores = {course_id: 0.0 for course_id in COURSE_IDS}
    explicitly_scored = set()
    for item in parsed.get("course_attitudes", []):
        course_id = int(item["course_id"])
        attitude = str(item["attitude"]).strip().upper().replace(" ", "_")
        attitude = ATTITUDE_ALIASES.get(attitude, attitude)
        if course_id not in COURSE_IDS:
            raise ValueError(f"Unknown course ID: {course_id}")
        if attitude not in ATTITUDE_VALUES:
            raise ValueError(f"Unknown attitude: {attitude}")
        course_scores[course_id] = ATTITUDE_VALUES[attitude]
        explicitly_scored.add(course_id)

    complements = sorted(set(canonical_pair(x) for x in parsed.get("complements", [])))
    substitutes = sorted(set(canonical_pair(x) for x in parsed.get("substitutes", [])))
    overlap = set(complements) & set(substitutes)
    if overlap:
        # Ambiguous relationships are conservatively discarded.
        complements = [pair for pair in complements if pair not in overlap]
        substitutes = [pair for pair in substitutes if pair not in overlap]

    return {
        "course_scores": course_scores,
        "explicitly_scored_courses": sorted(explicitly_scored),
        "complements": complements,
        "substitutes": substitutes,
    }


@torch.inference_mode()
def generate_profile(report, max_new_tokens=420):
    payload = {"course_catalog": PUBLIC_CATALOG, "student_report": report}
    messages = [
        {"role": "system", "content": PROFILE_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = extract_json_object(raw)
    return validate_profile(parsed), raw

## 5. Smoke test

In [ ]:
smoke_student_id = int(reports_df.student_id.iloc[0])
smoke_profile, smoke_raw = generate_profile(reports[smoke_student_id])
print("Raw output:")
print(smoke_raw)
print("\nValidated profile:")
print(smoke_profile)

## 6. Extract one profile per student with checkpointing

In [ ]:
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            completed[int(item["student_id"])] = item
print("Loaded checkpoint rows:", len(completed))

for student_id in tqdm(sorted(map(int, reports)), desc="Extracting profiles"):
    if student_id in completed and completed[student_id].get("status") == "ok":
        continue
    started = time.perf_counter()
    try:
        profile, raw = generate_profile(reports[student_id])
        item = {
            "student_id": student_id,
            "status": "ok",
            "profile": profile,
            "raw_output": raw,
            "error": None,
            "seconds": time.perf_counter() - started,
        }
    except Exception as exc:
        item = {
            "student_id": student_id,
            "status": "error",
            "profile": None,
            "raw_output": None,
            "error": repr(exc),
            "seconds": time.perf_counter() - started,
        }
    completed[student_id] = item
    with CHECKPOINT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item) + "\n")

status_df = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"profile", "raw_output"}}
    for item in completed.values()
]).sort_values("student_id")
display(status_df)
print(status_df.status.value_counts())

failed = status_df.loc[status_df.status != "ok", "student_id"].tolist()
assert not failed, f"Profile extraction failed for students: {failed}. Re-run this cell to retry them."

## 7. Convert profiles and bundle pairs into model features

In [ ]:
profiles = {student_id: item["profile"] for student_id, item in completed.items()}


def bundle_features(profile, bundle):
    bundle = tuple(sorted(map(int, bundle)))
    pairs = set(combinations(bundle, 2))
    score_sum = sum(float(profile["course_scores"].get(str(j), profile["course_scores"].get(j, 0.0))) for j in bundle)
    complement_count = sum(tuple(pair) in pairs for pair in profile["complements"])
    substitute_count = sum(tuple(pair) in pairs for pair in profile["substitutes"])
    explicit_count = sum(j in set(profile["explicitly_scored_courses"]) for j in bundle)
    return np.array([
        score_sum / MAX_BUNDLE_SIZE,
        complement_count / MAX_BUNDLE_SIZE,
        substitute_count / MAX_BUNDLE_SIZE,
        explicit_count / MAX_BUNDLE_SIZE,
    ], dtype=float)


feature_rows = []
for row in queries.itertuples(index=False):
    profile = profiles[int(row.student_id)]
    feature_difference = bundle_features(profile, row.bundle_a) - bundle_features(profile, row.bundle_b)
    feature_rows.append(feature_difference)

X = np.vstack(feature_rows)
FEATURE_NAMES = [
    "course_score_difference",
    "complement_difference",
    "substitute_difference",
    "explicit_coverage_difference",
]
for index, name in enumerate(FEATURE_NAMES):
    queries[name] = X[:, index]

display(queries.head())

The profile-extraction checkpoint is shared by filename across the two notebooks. If both notebooks run in the same Colab workspace, the second notebook reuses the completed LLM profiles rather than calling the GPU again.


## 8. Continuous score model, Approx. CEEI, and evaluation metrics


In [ ]:
N_REPEATED_SPLITS = 20
N_TRAIN_STUDENTS = 96
N_TEST_STUDENTS = 32
N_CEEI_RUNS = 10
MARKET_CAPACITY_PER_COURSE = 5
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]

EQUAL_BUDGET = 1.0
PRICE_ITERATIONS = 200
PRICE_RESTARTS = 3
PRICE_STEP = 0.08
PRICE_FLOOR = 1e-4

market_capacities = {int(c): MARKET_CAPACITY_PER_COURSE for c in COURSE_IDS}


def fit_cardinal_score_model(rows, return_cv=False):
    X = rows[FEATURE_NAMES].to_numpy()
    y = rows.true_difference.to_numpy()
    groups = rows.student_id.to_numpy()
    cv = GroupKFold(n_splits=min(N_FOLDS, rows.student_id.nunique()))
    results = []
    for alpha in ALPHAS:
        oof = np.full(len(rows), np.nan)
        for fit_idx, hold_idx in cv.split(X, y, groups):
            model = Ridge(alpha=alpha).fit(X[fit_idx], y[fit_idx])
            oof[hold_idx] = model.predict(X[hold_idx])
        results.append({"alpha": alpha, "oof_mse": float(np.mean((oof - y) ** 2))})
    cv_results = pd.DataFrame(results).sort_values(["oof_mse", "alpha"]).reset_index(drop=True)
    best_alpha = float(cv_results.loc[0, "alpha"])
    model = Ridge(alpha=best_alpha).fit(X, y)
    return (model, best_alpha, cv_results) if return_cv else model


def build_market_scores(rows, model):
    predicted, truth = {}, {}
    for student_id, student_rows in rows.groupby("student_id"):
        i = int(student_id)
        bundles = {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_a
        } | {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_b
        }
        predicted[i] = {
            b: float(np.dot(model.coef_, bundle_features(profiles[i], b)))
            for b in bundles
        }
        observations = {b: [] for b in bundles}
        for row in student_rows.itertuples(index=False):
            a = tuple(sorted(map(int, row.bundle_a)))
            b = tuple(sorted(map(int, row.bundle_b)))
            observations[a].append(float(row.true_utility_a))
            observations[b].append(float(row.true_utility_b))
        truth[i] = {b: float(np.mean(v)) for b, v in observations.items()}
        predicted[i][tuple()] = min(predicted[i].values()) - 2.0
        truth[i][tuple()] = min(truth[i].values()) - TRUE_COMMON_TOLERANCE - 1.0
    return predicted, truth


def make_tier_ranks(score_map, tolerance):
    ordered = sorted(score_map, key=lambda b: (-score_map[b], b))
    ranks, tier, anchor = {}, -1, None
    for bundle in ordered:
        score = score_map[bundle]
        if anchor is None or anchor - score > tolerance:
            tier += 1; anchor = score
        ranks[bundle] = tier
    return ranks


def fits(bundle, remaining):
    return all(remaining[c] > 0 for c in bundle)


def bundle_cost(bundle, prices):
    return sum(float(prices[c]) for c in bundle)


def choose_by_score(i, candidates, scores, priority):
    candidates = list(candidates)
    best = max(scores[i][b] for b in candidates)
    top = [b for b in candidates if np.isclose(scores[i][b], best, atol=1e-12, rtol=0)]
    return min(top, key=lambda b: priority[(i, b)])


def make_priorities(score_maps, seed):
    rng = np.random.default_rng(seed)
    return {(i, b): float(rng.random()) for i in sorted(score_maps) for b in sorted(score_maps[i])}


def run_approx_ceei(predicted_scores, true_scores, seed, oracle=False):
    students = sorted(predicted_scores)
    demand_scores = true_scores if oracle else predicted_scores
    master_rng = np.random.default_rng(seed)
    best_state = None
    for _ in range(PRICE_RESTARTS):
        restart_seed = int(master_rng.integers(2**32 - 1))
        rng = np.random.default_rng(restart_seed)
        prices = {c: float(rng.uniform(0.02, 0.10)) for c in COURSE_IDS}
        priority = make_priorities(predicted_scores, restart_seed + 1)
        for iteration in range(PRICE_ITERATIONS):
            demand = {}
            for i in students:
                affordable = [b for b in predicted_scores[i] if bundle_cost(b, prices) <= EQUAL_BUDGET + 1e-10]
                demand[i] = choose_by_score(i, affordable, demand_scores, priority)
            counts = Counter(c for b in demand.values() for c in b)
            excess = {c: counts[c] - market_capacities[c] for c in COURSE_IDS}
            step = PRICE_STEP / np.sqrt(iteration + 1)
            for c in COURSE_IDS:
                prices[c] = max(PRICE_FLOOR, prices[c] + step * excess[c] / max(1, market_capacities[c]))
        demand = {}
        for i in students:
            affordable = [b for b in predicted_scores[i] if bundle_cost(b, prices) <= EQUAL_BUDGET + 1e-10]
            demand[i] = choose_by_score(i, affordable, demand_scores, priority)
        counts = Counter(c for b in demand.values() for c in b)
        excess = {c: counts[c] - market_capacities[c] for c in COURSE_IDS}
        clearing_loss = float(sum(max(0, x) ** 2 for x in excess.values()))
        if best_state is None or clearing_loss < best_state[0]:
            best_state = (clearing_loss, prices.copy(), priority)

    clearing_loss, prices, priority = best_state
    rng = np.random.default_rng(seed + 991)
    order = list(rng.permutation(students))
    remaining = market_capacities.copy()
    allocation = {}
    for i in order:
        affordable = [b for b in predicted_scores[i] if bundle_cost(b, prices) <= EQUAL_BUDGET + 1e-10]
        feasible_affordable = [b for b in affordable if fits(b, remaining)]
        feasible_any = [b for b in predicted_scores[i] if fits(b, remaining)]
        pool = feasible_affordable if feasible_affordable else feasible_any
        chosen = choose_by_score(i, pool, demand_scores, priority)
        allocation[i] = chosen
        for c in chosen:
            remaining[c] -= 1
    return allocation, clearing_loss


def max_pareto_improvements(allocation, true_scores):
    true_ranks = {i: make_tier_ranks(true_scores[i], TRUE_COMMON_TOLERANCE) for i in true_scores}
    students = sorted(true_ranks)
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in students
    }
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    rows, lo, hi = [], [], []
    for i in students:
        row = {index[(i, b)]: 1.0 for b in allowed[i]}
        rows.append(row); lo.append(1.0); hi.append(1.0)
    for course in COURSE_IDS:
        row = {index[(i, b)]: 1.0 for i, b in variables if course in b}
        rows.append(row); lo.append(-np.inf); hi.append(float(market_capacities[course]))
    matrix = lil_matrix((len(rows), len(variables)), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items(): matrix[r, col] = value
    objective = np.array([
        -float(true_ranks[i][b] < true_ranks[i][allocation[i]]) for i, b in variables
    ])
    result = milp(
        c=objective, integrality=np.ones(len(variables)),
        bounds=Bounds(np.zeros(len(variables)), np.ones(len(variables))),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lo), np.array(hi)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        return np.nan
    chosen = {i: b for (i, b), x in zip(variables, result.x) if x > 0.5}
    return int(sum(true_ranks[i][chosen[i]] < true_ranks[i][allocation[i]] for i in students))


def envy_metrics(allocation, true_scores):
    true_ranks = {i: make_tier_ranks(true_scores[i], TRUE_COMMON_TOLERANCE) for i in true_scores}
    envy, evaluable = 0, 0
    for i in allocation:
        for j in allocation:
            if i == j or allocation[j] not in true_ranks[i]:
                continue
            evaluable += 1
            envy += true_ranks[i][allocation[j]] < true_ranks[i][allocation[i]]
    rate = float(envy / evaluable) if evaluable else np.nan
    return {"envy_pairs": int(envy), "evaluable_pairs": int(evaluable),
            "envy_rate": rate, "ef_score": 1.0 - rate if np.isfinite(rate) else np.nan}


def evaluate_ceei(allocation, true_scores):
    pe_count = max_pareto_improvements(allocation, true_scores)
    metrics = {
        "pareto_improvable_students": pe_count,
        "pareto_efficient": int(pe_count == 0) if np.isfinite(pe_count) else np.nan,
        "pe_score": 1.0 - pe_count / len(allocation) if np.isfinite(pe_count) else np.nan,
    }
    metrics.update(envy_metrics(allocation, true_scores))
    return metrics


## 9. Repeated held-out CEEI evaluation


In [ ]:
all_ids = np.array(sorted(queries.student_id.unique()))
assert len(all_ids) == N_TRAIN_STUDENTS + N_TEST_STUDENTS
split_rng = np.random.default_rng(SEED + 900)

manifest_rows, run_rows = [], []
for split_id in tqdm(range(N_REPEATED_SPLITS), desc="CEEI repeated train-test splits"):
    shuffled = split_rng.permutation(all_ids)
    train_ids = set(map(int, shuffled[:N_TRAIN_STUDENTS]))
    test_ids = set(map(int, shuffled[N_TRAIN_STUDENTS:]))
    for role, ids in [("train", train_ids), ("test", test_ids)]:
        manifest_rows.extend({"split_id": split_id, "student_id": i, "role": role} for i in ids)

    train = queries[queries.student_id.isin(train_ids)].copy()
    test = queries[queries.student_id.isin(test_ids)].copy()
    model, alpha, _ = fit_cardinal_score_model(train, return_cv=True)
    predicted, truth = build_market_scores(test, model)

    for run in range(N_CEEI_RUNS):
        paired = {}
        for oracle, method in [(False, "continuous_proxy"), (True, "oracle_true_utility")]:
            allocation, clearing_loss = run_approx_ceei(
                predicted, truth, seed=SEED + 100_000 * split_id + run, oracle=oracle
            )
            metrics = evaluate_ceei(allocation, truth)
            paired[method] = metrics
            run_rows.append({
                "split_id": split_id, "run": run, "method": method,
                "ridge_alpha": alpha, "clearing_loss": clearing_loss, **metrics,
            })

results = pd.DataFrame(run_rows)

# Pair proxy and oracle runs to compute oracle-normalized scores and gaps.
wide = results.pivot(index=["split_id", "run"], columns="method",
                     values=["pe_score", "ef_score", "envy_rate"]).reset_index()
wide.columns = ["_".join([str(x) for x in col if str(x)]) for col in wide.columns]
wide["pe_score_ratio"] = (
    wide["pe_score_continuous_proxy"] / wide["pe_score_oracle_true_utility"].replace(0, np.nan)
)
wide["ef_score_ratio"] = (
    wide["ef_score_continuous_proxy"] / wide["ef_score_oracle_true_utility"].replace(0, np.nan)
)
wide["excess_envy"] = (
    wide["envy_rate_continuous_proxy"] - wide["envy_rate_oracle_true_utility"]
)

summary = (
    results.groupby("method", as_index=False)
    .agg(
        pe_rate=("pareto_efficient", "mean"),
        mean_pe_score=("pe_score", "mean"),
        mean_ef_score=("ef_score", "mean"),
        mean_envy_rate=("envy_rate", "mean"),
        mean_clearing_loss=("clearing_loss", "mean"),
        worst_pe_score=("pe_score", "min"),
        worst_ef_score=("ef_score", "min"),
    )
)
ratio_summary = wide[["pe_score_ratio", "ef_score_ratio", "excess_envy"]].describe().T
display(summary)
display(ratio_summary)


## 10. Visualize and save CEEI results


In [ ]:
FIGURE_DIR = Path("ceei_efficiency_envy_figures")
FIGURE_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
axes[0].hist(wide.pe_score_ratio.dropna(), bins=20, color="#2563eb", alpha=0.8)
axes[0].axvline(1, color="black", linestyle="--")
axes[0].set(title="Oracle-normalized PE score", xlabel="Proxy / oracle", ylabel="Count")
axes[1].hist(wide.ef_score_ratio.dropna(), bins=20, color="#f59e0b", alpha=0.8)
axes[1].axvline(1, color="black", linestyle="--")
axes[1].set(title="Oracle-normalized EF score", xlabel="Proxy / oracle")
axes[2].hist(wide.excess_envy.dropna(), bins=20, color="#dc2626", alpha=0.75)
axes[2].axvline(0, color="black", linestyle="--")
axes[2].set(title="Excess true-preference envy", xlabel="Proxy envy - oracle envy")
for ax in axes: ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "ceei_oracle_comparison.png", dpi=180)
plt.show()

OUTPUT_DIR = Path("ceei_efficiency_envy_results")
OUTPUT_DIR.mkdir(exist_ok=True)
pd.DataFrame(manifest_rows).to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
results.to_csv(OUTPUT_DIR / "allocation_runs.csv", index=False)
wide.to_csv(OUTPUT_DIR / "paired_oracle_ratios.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
ratio_summary.to_csv(OUTPUT_DIR / "ratio_summary.csv")
for path in FIGURE_DIR.glob("*.png"):
    shutil.copy2(path, OUTPUT_DIR / path.name)

metadata = {
    "seed": SEED,
    "mechanism": "Approx_CEEI continuous-score price search plus feasibility repair",
    "n_repeated_splits": N_REPEATED_SPLITS,
    "split_sizes": {"train": N_TRAIN_STUDENTS, "test": N_TEST_STUDENTS},
    "capacity_per_course": MARKET_CAPACITY_PER_COURSE,
    "oracle": "same Approx_CEEI algorithm using true bundle utility",
    "scope_warning": "This implementation does not certify an exact competitive equilibrium.",
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR))
print("Saved:", archive.resolve())
from google.colab import files
files.download(str(archive))
